# Kwartalny wstępny rachunek zysków i strat z użyciem PROC COMPUTAB

## Streszczenie kierownicze

Ten notatnik tworzy kwartalny wstępny (pro forma) rachunek zysków i strat regionalnego banku bezpośrednio z miesięcznych danych księgi głównej za pomocą PROC COMPUTAB — procedury tabelarycznego raportowania SAS/ETS. Kieruje przychody odsetkowe, koszty odsetkowe, przychody z opłat oraz koszty operacyjne każdego miesiąca do właściwej kolumny kwartału kalendarzowego, a następnie za pomocą bloków programowania wierszy oblicza wynik z tytułu odsetek netto, dochód przed opodatkowaniem i dochód netto, natomiast blok kolumn sumuje cztery kwartały w łączną wartość roczną.

## Źródła danych

Jeden syntetyczny zbiór danych `bank_ledger` symuluje jeden rok obrotowy miesięcznych pozycji sprawozdania finansowego dla średniej wielkości banku spółdzielczego. Dwanaście miesięcznych obserwacji jest generowanych w miejscu za pomocą `call streaminit`/`rand`, dzięki czemu notatnik jest w pełni samowystarczalny.

| Zmienna | Typ | Opis |
|----------|------|-------------|
| `stmtdate` | Num (DATE9.) | Data sprawozdania na koniec miesiąca (sty–gru) |
| `loanint`  | Num | Odsetki i opłaty uzyskane z portfela kredytowego (tys. USD) |
| `secint`   | Num | Odsetki uzyskane z portfela papierów wartościowych (tys. USD) |
| `depint`   | Num | Odsetki zapłacone od depozytów (tys. USD) |
| `borrint`  | Num | Odsetki zapłacone od zobowiązań / zaliczek FHLB (tys. USD) |
| `feeinc`   | Num | Przychody pozaodsetkowe (opłaty i prowizje za usługi) (tys. USD) |
| `salaries` | Num | Koszty wynagrodzeń i świadczeń pracowniczych (tys. USD) |
| `occupancy`| Num | Koszty utrzymania nieruchomości i wyposażenia (tys. USD) |
| `othropex` | Num | Pozostałe pozaodsetkowe koszty operacyjne (tys. USD) |
| `provision`| Num | Rezerwa na straty kredytowe (tys. USD) |
| `taxrate`  | Num | Efektywna stawka podatkowa stosowana do dochodu przed opodatkowaniem |

# Kwartalny wstępny rachunek zysków i strat z użyciem PROC COMPUTAB

Zespoły finansowe banków rutynowo przekształcają miesięczną księgę główną w **kwartalny rachunek zysków i strat**, który pokazuje wynik z tytułu odsetek netto oraz wynik netto na dole zestawienia. `PROC COMPUTAB` (SAS/ETS) jest do tego stworzony: układa programowalną tabelę, której **kolumny** są okresami sprawozdawczymi, a **wiersze** pozycjami zestawienia, i pozwala obliczać podsumowania za pomocą zwykłej logiki kroku DATA w blokach wierszy i kolumn.

W tym notatniku:

1. Generujemy jeden rok obrotowy syntetycznych miesięcznych danych księgi głównej banku spółdzielczego.
2. Kierujemy każdy miesiąc do jego kwartału kalendarzowego i budujemy kwartalny rachunek zysków i strat.
3. Obliczamy wynik z tytułu odsetek netto, dochód przed opodatkowaniem i dochód netto w **bloku wierszy**, a następnie sumujemy kwartały w łączną wartość roczną w **bloku kolumn**.
4. Ponownie wykorzystujemy tabelę `out=`, aby obliczone zestawienie mogło zasilać dalsze raportowanie.

## Krok 1 — Wygenerowanie syntetycznych miesięcznych danych księgi głównej

Symulujemy dwanaście obserwacji na koniec miesiąca. Przychody odsetkowe z kredytów i papierów wartościowych łagodnie rosną w ciągu roku, koszty depozytów i zobowiązań skalują się wraz z otoczeniem stóp procentowych, a przychody z opłat, koszty operacyjne i rezerwa na straty kredytowe niosą realistyczny szum z miesiąca na miesiąc. `call streaminit` ustala ziarno, dzięki czemu dane są odtwarzalne.

In [1]:
DANE bank_ledger;
   CALL streaminit(20240115);
   format stmtdate date9.;
   POWTÓRZ month = 1 TO 12;
      /* Month-end statement date for fiscal year 2024 */
      stmtdate = mdy(month, 28, 2024);

      /* Mild upward drift across the year + monthly noise */
      drift = 1 + 0.012 * (month - 1);

      /* Interest income (USD thousands) */
      loanint = round(1850 * drift + rand('normal', 0, 45), 0.01);
      secint  = round( 420 * drift + rand('normal', 0, 15), 0.01);

      /* Interest expense (USD thousands) */
      depint  = round( 540 * drift + rand('normal', 0, 22), 0.01);
      borrint = round( 130 * drift + rand('normal', 0, 10), 0.01);

      /* Non-interest income and expense (USD thousands) */
      feeinc    = round(310 + rand('normal', 0, 18), 0.01);
      salaries  = round(720 + rand('normal', 0, 25), 0.01);
      occupancy = round(165 + rand('normal', 0, 8),  0.01);
      othropex  = round(240 + rand('normal', 0, 20), 0.01);

      /* Provision for credit losses, occasionally elevated */
      provision = round(95 + rand('exponential') * 40, 0.01);

      /* Effective tax rate */
      taxrate = 0.21;

      WYJŚCIE;
   KONIEC;
   ZACHOWAJ stmtdate loanint secint depint borrint
        feeinc salaries occupancy othropex provision taxrate;
WYKONAJ;

PROCEDURA DRUKUJ DANE=bank_ledger noobs ETYKIETA;
   TYTUŁ 'Synthetic Monthly Ledger — Community Bank FY2024';
   format loanint secint depint borrint feeinc
          salaries occupancy othropex provision comma8.2;
WYKONAJ;

                                    Synthetic Monthly Ledger — Community Bank FY2024                                    

 STMTDATE   LOANINT  SECINT  DEPINT  BORRINT  FEEINC  SALARIES  OCCUPANCY  OTHROPEX  PROVISION  TAXRATE
28JAN2024  1,772.95  423.79  531.47   128.99  339.33    699.38     171.95    202.43     130.93     0.21
28FEB2024  1,824.38  421.13  564.85   125.53  294.09    767.29     162.79    307.61     123.25     0.21
28MAR2024  1,931.98  442.06  536.64   131.71  295.72    724.03     153.26    254.21     115.76     0.21
28APR2024  1,934.99  439.29  535.94   140.01  294.56    729.47     174.19    237.43     198.05     0.21
28MAY2024  1,949.31  447.35  591.44   124.42  299.78    739.13     165.08    223.32     105.57     0.21
28JUN2024  1,934.36  448.28  551.70   147.64  335.81    718.90     171.91    236.94     130.13     0.21
28JUL2024  1,936.57  439.22  565.70   133.82  293.21    743.87     174.16    247.19     199.20     0.21
28AUG2024  1,979.73  457.42  558.54   144.45  

## Krok 2 — Zbudowanie kwartalnego rachunku zysków i strat

Sercem raportu jest poniższy krok `PROC COMPUTAB`.

- **`columns qtr1 qtr2 qtr3 qtr4 fy;`** definiuje cztery kolumny kwartałów oraz kolumnę pełnego roku.
- Nieoznaczony **blok wejściowy** ustawia zmienną automatyczną **`_col_`** za pomocą `qtr(stmtdate)`, co kieruje każdą miesięczną obserwację do właściwej kolumny kwartału. Ponieważ dane wejściowe są domyślnie transponowane, każda zmienna księgi trafia do wiersza o tej samej nazwie.
- **Blok wierszy** `inc_stmt:` uruchamia się raz na kolumnę i oblicza pozycje pochodne — wynik z tytułu odsetek netto, łączne koszty pozaodsetkowe, dochód przed opodatkowaniem i dochód netto — dokładnie tak, jak zrobiłby to księgowy.
- **Blok kolumn** `total:` uruchamia się raz na wiersz i sumuje cztery kwartały do kolumny `fy` (pełny rok).

Instrukcje `rows ... / ...` dodają tytuły, wcięcia i linie reguł (`ol` nadkreślenie, `ul` podkreślenie, `dul` podwójne podkreślenie), dzięki czemu wynik czyta się jak prawdziwe sprawozdanie finansowe.

In [2]:
TYTUŁ 'Pro Forma Income Statement — Community Bancorp, Inc.';
title2 'Fiscal Year 2024  (Amounts in USD Thousands)';

PROCEDURA computab DANE=bank_ledger cspace=2 cwidth=11 out=qtr_income;

   /* Four quarters plus a rolled-up full-year column */
   columns qtr1 qtr2 qtr3 qtr4 fy / format=comma11.1;
   columns qtr1 / 'Q1';
   columns qtr2 / 'Q2';
   columns qtr3 / 'Q3';
   columns qtr4 / 'Q4';
   columns fy   / 'Full' 'Year' +3;

   /* Income statement rows, top to bottom */
   rows loanint  / 'Interest & Fees on Loans';
   rows secint   / 'Interest on Securities'        ul;
   rows intinc   / 'Total Interest Income';
   rows depint   / 'Interest on Deposits';
   rows borrint  / 'Interest on Borrowings'        ul;
   rows intexp   / 'Total Interest Expense';
   rows nii      / 'Net Interest Income'           ol skip;
   rows provision/ 'Provision for Credit Losses'   ul;
   rows niiap    / 'Net Int. Income after Prov.'   skip;
   rows feeinc   / 'Non-Interest Income'           ul;
   rows salaries / 'Salaries & Benefits';
   rows occupancy/ 'Occupancy & Equipment';
   rows othropex / 'Other Operating Expense'       ul;
   rows nonintexp/ 'Total Non-Interest Expense'    skip;
   rows pretax   / 'Pre-Tax Income'                ol;
   rows taxexp   / 'Income Tax Expense'            ul;
   rows netinc   / 'Net Income'                    dul;

   /* Input block: route each month to its calendar quarter */
   _col_ = qtr(stmtdate);

   /* Row block: compute statement subtotals across every column */
   inc_stmt:
      intinc    = loanint + secint;
      intexp    = depint + borrint;
      nii       = intinc - intexp;
      niiap     = nii - provision;
      nonintexp = salaries + occupancy + othropex;
      pretax    = niiap + feeinc - nonintexp;
      taxexp    = pretax * 0.21;
      netinc    = pretax - taxexp;

   /* Column block: roll the four quarters into the full-year column */
   TOTAL:
      fy = qtr1 + qtr2 + qtr3 + qtr4;
WYKONAJ;

                                  Pro Forma Income Statement — Community Bancorp, Inc.                                  
                                      Fiscal Year 2024  (Amounts in USD Thousands)                                      


                             The COMPUTAB Procedure                             

                             Q1           Q2           Q3           Q4         Full  
                                                                               Year  
                           qtr1         qtr2         qtr3         qtr4           fy  
                    -----------  -----------  -----------  -----------  -----------  
  Interest & Fees on Loans
  loanint               5529.31      5818.66      5963.46      6276.96     23588.39  
  Interest on Securities
  secint                1286.98      1334.92      1342.03      1452.88      5416.81  
                    -----------  -----------  -----------  -----------  -----------  
  Total Interest Inc

## Krok 3 — Ponowne wykorzystanie zbioru wynikowego COMPUTAB

Powyższy krok `PROC COMPUTAB` zapisał obliczoną tabelę do `qtr_income` za pomocą `out=`. Każdy wiersz tego zbioru danych jest pozycją zestawienia, a każda zmienna kolumnowa (`qtr1`–`qtr4`, `fy`) przechowuje obliczoną wartość, więc może zasilać dalsze raportowanie. Poniżej drukujemy zsumowaną kolumnę pełnego roku, aby potwierdzić, że liczby się zgadzają.

In [3]:
TYTUŁ 'COMPUTAB Output Dataset — Full-Year Column';

PROCEDURA DRUKUJ DANE=qtr_income ETYKIETA noobs;
   ZMIENNA _row_ fy;
   format fy comma12.1;
   ETYKIETA _row_ = 'Line Item' fy = 'Full Year';
WYKONAJ;

TYTUŁ;

                                       COMPUTAB Output Dataset — Full-Year Column                                       
                                      Fiscal Year 2024  (Amounts in USD Thousands)                                      


Line Item  Full Year
---------  ---------
loanint     23,588.4
secint       5,416.8
intinc      29,005.2
depint       6,831.2
borrint      1,650.7
intexp       8,482.0
nii         20,523.2
provision    1,568.9
niiap       18,954.3
feeinc       3,703.2
salaries     8,763.1
occupancy    1,985.6
othropex     2,944.8
nonintexp   13,693.5
pretax       8,964.1
taxexp       1,882.5
netinc       7,081.6

NOTE: Option TITLE changed to COMPUTAB Output Dataset — Full-Year Column.
NOTE: PROC PRINT data=qtr_income

NOTE: PROC PRINT completed: 17 observations printed, 2 variables


## Interpretacja wyników

Wstępny rachunek zysków i strat czyta się od góry do dołu jak regulacyjny raport sprawozdawczy (call report) banku: łączne przychody odsetkowe pomniejszone o łączne koszty odsetkowe dają **wynik z tytułu odsetek netto** — tutaj około 20,5 mln USD za rok — główny czynnik zysków banku. Odjęcie **rezerwy na straty kredytowe**, dodanie **przychodów z opłat** i odjęcie **kosztów pozaodsetkowych** daje dochód przed opodatkowaniem w wysokości około 9,0 mln USD, a zastosowanie efektywnej stawki podatkowej 21% daje **dochód netto** bliski 7,1 mln USD. Blok kolumn `total:` sumuje cztery kwartały do kolumny pełnego roku, więc łączne wartości roczne z założenia uzgadniają się z sumą kwartałów — co potwierdza zbiór `out=`, w którym roczny `netinc` wynoszący 7 081,6 równa się sumie czterech wartości kwartalnych.

Ponieważ wszystko jest obliczane wewnątrz programowalnych bloków wierszy i kolumn `PROC COMPUTAB`, podstawienie prawdziwej miesięcznej księgi głównej nie wymaga żadnej zmiany logiki raportu — zmienia się jedynie zbiór wejściowy. Zbiór `out=` udostępnia następnie obliczone zestawienie do pulpitów menedżerskich, analizy trendów lub automatyzacji pakietów dla zarządu.